# 01. Librerie

In [12]:
%reset -f
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t as student_t
import re

# 02. Input

In [13]:
# input / output
input = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\99_Modelli\03.output"
output = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\99_Modelli\03.output_test"
MACRO = pd.read_excel(input + r"\01_Macrovar&Target - no AGG.xlsx")

# 03. Definizione modelli

## Regressioni

In [14]:
ATECO = [f"_{i}" for i in range(1, 16)]

coe_rows = []
pred_rows = []

for cod_ateco in ATECO:
    print("ATECO:", cod_ateco)
    # colonne che finiscono con _k
    cols_group = [c for c in MACRO.columns if c.endswith(cod_ateco)]

    MACRO_ateco = MACRO[["Anno"] + cols_group].copy()

    targets = [c for c in MACRO_ateco.columns if (("TD" in c) and c.endswith(cod_ateco) and not c.startswith("L1_TD") and not c.startswith("L2_TD"))]

    for target_select in targets:

        # Predittori X:
        names_ateco = [c for c in MACRO_ateco.columns if c not in ["Anno", target_select] and ("TD" not in c)]

        long_df = MACRO_ateco[["Anno", target_select] + names_ateco].melt(id_vars=["Anno", target_select],
            value_vars=names_ateco,
            var_name="Variabile",
            value_name="Valore"
        )

        for nm in names_ateco:

            dati = long_df[long_df["Variabile"] == nm].copy()

            y = dati[target_select].astype(float)
            x = dati["Valore"].astype(float)

            # correlazione
            ok = x.notna() & y.notna()
            n = int(ok.sum())

            if n < 3:
                corr = np.nan
                pcorr = np.nan
            else:
                corr = np.corrcoef(x[ok], y[ok])[0, 1]
                if np.isfinite(corr) and abs(corr) < 1:
                    t_stat = corr * np.sqrt((n - 2) / (1 - corr**2))
                    pcorr = 2 * (1 - student_t.cdf(abs(t_stat), df=n - 2))
                else:
                    pcorr = np.nan

            # OLS con intercetta
            X = sm.add_constant(x)
            model = sm.OLS(y, X, missing="drop").fit()

            if "Valore" in model.params.index:
                beta = float(model.params["Valore"])
                tstat = float(model.tvalues["Valore"])
                pval = float(model.pvalues["Valore"])
            else:
                beta = np.nan
                tstat = np.nan
                pval = np.nan

            r2 = model.rsquared
            r2a = model.rsquared_adj

            coe_rows.append({
                "ATECO": cod_ateco.replace("_", ""),
                "TARGET": target_select,
                "VAR": nm,
                "BETA": beta,
                "TSTAT": np.round(tstat, 3),
                "PVAL": np.round(pval, 4),
                "R2": np.round(r2, 4),
                "R2_ADJ": np.round(r2a, 4),
                "CORR": np.round(corr, 4),
                "PCORR": np.round(pcorr, 4),
            })

            # predizioni
            yhat = model.predict(X)
            for a, xv, yv, yh in zip(dati["Anno"], x, y, yhat):
                pred_rows.append({
                    "Anno": a,
                    "TARGET": target_select,
                    "VAR": nm,
                    "X": xv,
                    "Y": yv,
                    "YHAT": yh,
                    "ATECO": cod_ateco.replace("_", "")
                })

COE_ateco = pd.DataFrame(coe_rows)
PREDCU_ateco = pd.DataFrame(pred_rows)


ATECO: _1
ATECO: _2
ATECO: _3
ATECO: _4
ATECO: _5
ATECO: _6
ATECO: _7
ATECO: _8
ATECO: _9
ATECO: _10
ATECO: _11
ATECO: _12
ATECO: _13
ATECO: _14
ATECO: _15


## Definizione VAR2 per cluster più larghi di variabile

In [15]:
allowed = ["infortuni","mortali","tasso_infortuni","tasso_mortali","incidenza_mortali"]

def make_var2(v):

    if pd.isna(v):
        return pd.NA

    # rimuovo suffisso _ATECO
    v0 = re.sub(r"_\d+$", "", v)

    # caso L1_
    if v0.startswith("L1_"):
        base = v0[3:]
        if base in allowed:
            return "L1_" + base
        else:
            return pd.NA

    # caso livello
    if v0 in allowed:
        return v0

    return pd.NA

COE_ateco["VAR2"] = COE_ateco["VAR"].apply(make_var2)

## Flag e outcome modelli

In [16]:
# ---------- FLAG ----------
OUT_UNI = COE_ateco.copy()

OUT_UNI["FL_PVALUE"] = np.where( OUT_UNI["PVAL"].notna() & (OUT_UNI["PVAL"] <= 0.1), 0, 1)
OUT_UNI["FL_PCORR"]  = np.where(OUT_UNI["PCORR"].notna() & (OUT_UNI["PCORR"] <= 0.05), 0, 1)
OUT_UNI["FL_R2"]     = np.where(OUT_UNI["R2_ADJ"].notna() & (OUT_UNI["R2_ADJ"] > 0.005), 0, 1)

# segni attesi
segni_attesi = {
    "infortuni": "+",
    "mortali": "+",
    "tasso_infortuni": "+",
    "tasso_mortali": "+",
    "incidenza_mortali": "+",}

def expected_sign(var_name: str):
    for k, v in segni_attesi.items():
        if k in var_name:
            return v
    return None

OUT_UNI["SIGN_VAR_ATTESO"] = OUT_UNI["VAR"].apply(expected_sign)

def fl_sign(row):
    s = row["SIGN_VAR_ATTESO"]
    b = row["BETA"]
    if s is None or pd.isna(b):
        return pd.NA
    if s == "+" and b >= 0:
        return 0
    if s == "-" and b < 0:
        return 0
    return 1

OUT_UNI["FL_SIGN"] = OUT_UNI.apply(fl_sign, axis=1).astype("Int64")

# FLAG_TOT 
fl_sign_for_sum = OUT_UNI["FL_SIGN"].fillna(0).astype(int) #se NA conta 0
OUT_UNI["FLAG_TOT"] = np.where((OUT_UNI["FL_PVALUE"].astype(int) + OUT_UNI["FL_PCORR"].astype(int) + OUT_UNI["FL_R2"].astype(int) + fl_sign_for_sum) == 0,"OK","KO")

## Calcolo P-value  e R-squared adj  per Ateco e VAR2

In [26]:
app = OUT_UNI.copy()

#Solo TD in livelli
mask_td_level = app["TARGET"].astype(str).str.startswith("TD_")

#Filtro variabili selezionate
mask_x_simple = app["VAR"].astype(str).str.match(
    r"^(L1_)?(infortuni|mortali|tasso_infortuni|tasso_mortali|incidenza_mortali)_\d+$",
    na=False)

df_ok = app[
    mask_td_level &
    mask_x_simple &
    (app["FL_PVALUE"] == 0) &
    (app["FL_R2"] == 0) &
    (app["FL_SIGN"] == 0) &
    app["ATECO"].notna()].copy()


In [18]:
keep = ["VAR2", "ATECO", "CORR", "R2_ADJ", "PVAL"]
df_ok = df_ok[keep].copy()

# cast numerici
for c in ["CORR", "R2_ADJ", "PVAL"]:
    df_ok[c] = pd.to_numeric(df_ok[c], errors="coerce")

pred = PREDCU_ateco.copy()
pred["VAR2"] = pred["VAR"].apply(make_var2)
pred = pred[pred["VAR2"].notna() & pred["ATECO"].notna()].copy()
pred["X"] = pd.to_numeric(pred["X"], errors="coerce")

#Calcolo media variabile
x_mean = (
    pred.groupby(["VAR2", "ATECO"], as_index=False)
        .agg(X_MEAN=("X", "mean")))

res = df_ok.merge(x_mean, on=["VAR2", "ATECO"], how="left")

# Creazione tabella finale
mat_corr = res.pivot(index="VAR2", columns="ATECO", values="CORR")
mat_r2   = res.pivot(index="VAR2", columns="ATECO", values="R2_ADJ")
mat_pv   = res.pivot(index="VAR2", columns="ATECO", values="PVAL")
mat_xm   = res.pivot(index="VAR2", columns="ATECO", values="X_MEAN")

out_final = pd.concat(
    {"CORR": mat_corr, "R2_ADJ": mat_r2, "PVAL": mat_pv, "X_MEAN": mat_xm},
    names=["METRICA", "VAR2"]
).swaplevel(0, 1).sort_index()

# ordino colonne ATECO numericamente
out_final = out_final.reindex(sorted(out_final.columns, key=lambda x: int(x)), axis=1)

out_final

ATECO                                 1           2              3  \
VAR2               METRICA                                           
L1_infortuni       CORR        0.826900    0.585100       0.693200   
                   PVAL        0.000100    0.021900       0.004200   
                   R2_ADJ      0.659500    0.291800       0.440500   
                   X_MEAN   3086.176684  672.800000  103268.333333   
L1_mortali         CORR        0.626000    0.569900            NaN   
                   PVAL        0.012500    0.026500            NaN   
                   R2_ADJ      0.345100    0.272900            NaN   
                   X_MEAN     11.800000    5.200000            NaN   
L1_tasso_infortuni CORR        0.849300    0.562800       0.705100   
                   PVAL        0.000100    0.028900       0.003300   
                   R2_ADJ      0.699900    0.264200       0.458400   
                   X_MEAN      0.003539    0.030145       0.026267   
L1_tasso_mortali   CORR        0.672400    0.458100            NaN   
                   PVAL        0.006000    0.086000            NaN   
                   R2_ADJ      0.410000    0.149000            NaN   
                   X_MEAN      0.000013    0.000235            NaN   
infortuni          CORR        0.838900    0.477600       0.624900   
                   PVAL        0.000000    0.061400       0.009600   
                   R2_ADJ      0.682600    0.172900       0.347000   
                   X_MEAN   3192.437500  653.187500  102349.375000   
mortali            CORR        0.527200    0.464900            NaN   
                   PVAL        0.035900    0.069600            NaN   
                   R2_ADJ      0.226300    0.160200            NaN   
                   X_MEAN     11.562500    5.125000            NaN   
tasso_infortuni    CORR        0.855400    0.449400       0.649800   
                   PVAL        0.000000    0.080700       0.006400   
                   R2_ADJ      0.712600    0.145000       0.381000   
                   X_MEAN      0.003430    0.029473       0.026025   
tasso_mortali      CORR        0.565600         NaN            NaN   
                   PVAL        0.022400         NaN            NaN   
                   R2_ADJ      0.271300         NaN            NaN   
                   X_MEAN      0.000012         NaN            NaN   

ATECO                                 4         5         6             7  \
VAR2               METRICA                                                  
L1_infortuni       CORR        0.537700       NaN       NaN      0.661800   
                   PVAL        0.038700       NaN       NaN      0.007200   
                   R2_ADJ      0.234400       NaN       NaN      0.394800   
                   X_MEAN   1378.066667       NaN       NaN  52827.200000   
L1_mortali         CORR             NaN       NaN       NaN           NaN   
                   PVAL             NaN       NaN       NaN           NaN   
                   R2_ADJ           NaN       NaN       NaN           NaN   
                   X_MEAN           NaN       NaN       NaN           NaN   
L1_tasso_infortuni CORR        0.501600  0.510400  0.465400      0.680400   
                   PVAL        0.056800  0.051900  0.080400      0.005200   
                   R2_ADJ      0.194100  0.203700  0.156300      0.421700   
                   X_MEAN      0.016829  0.051658  0.029054      0.014221   
L1_tasso_mortali   CORR             NaN       NaN       NaN           NaN   
                   PVAL             NaN       NaN       NaN           NaN   
                   R2_ADJ           NaN       NaN       NaN           NaN   
                   X_MEAN           NaN       NaN       NaN           NaN   
infortuni          CORR        0.481500       NaN       NaN      0.545600   
                   PVAL        0.059000       NaN       NaN      0.028800   
                   R2_ADJ      0.176900       NaN       NaN      0.247500   
  

## Export

In [25]:
# ---------- EXPORT ----------
out_file = output + r"\02_Univariate_TD_livello.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    # Dataframe 
    OUT_UNI.to_excel(writer, sheet_name="UNIV_TD", index=False)

    #Resto
    PREDCU_ateco.to_excel(writer, sheet_name="PRED", index=False)
    out_final.to_excel(writer, sheet_name="Summary_modello", index=True)    